## 전처리

In [2]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [3]:
df = pd.read_csv("data/2025_Airbnb_NYC_listings.csv", index_col=0)
clean_df = df.copy()
print("="*60)
print("데이터 로드 완료!")
print("="*60)
print(f"Airbnb_data: {clean_df.shape}")


데이터 로드 완료!
Airbnb_data: (22308, 72)


In [4]:
target_cols = [
    "id","name","description","host_id","host_since","host_response_time",
    "host_response_rate","host_acceptance_rate","host_is_superhost",
    "neighbourhood_cleansed","neighbourhood_group_cleansed",
    "latitude","longitude","property_type","room_type","accommodates",
    "bedrooms","beds","amenities","price","availability_365",
    "number_of_reviews","number_of_reviews_ltm",
    "estimated_occupancy_l365d","estimated_revenue_l365d",
    "review_scores_rating","review_scores_accuracy","review_scores_cleanliness",
    "review_scores_checkin","review_scores_communication","review_scores_location",
    "review_scores_value",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",
    "reviews_per_month"
]

clean_df = df[target_cols]
clean_df.shape

(22308, 37)

In [5]:
# 'price' 1차 전처리
clean_df['price'] = pd.to_numeric(
    clean_df['price'].replace(r'[\$,]', '', regex=True), 
    errors='coerce'
)
clean_df['price']

0        200.0
1         82.0
2        765.0
3        139.0
4        130.0
         ...  
37429     72.0
37430     58.0
37431    299.0
37432    200.0
37433     58.0
Name: price, Length: 22308, dtype: float64

In [6]:
# 'bedrooms' 전처리
display(clean_df['bedrooms'].isna().sum())
display(clean_df['bedrooms'].value_counts(dropna=False))

np.int64(49)

bedrooms
1.0     14132
2.0      3863
0.0      2138
3.0      1444
4.0       449
5.0       156
6.0        53
NaN        49
7.0        13
9.0         5
8.0         4
14.0        1
15.0        1
Name: count, dtype: int64

In [7]:
clean_df.loc[clean_df["bedrooms"].isna(), ["host_id", "name", "price", "room_type", "beds", "bedrooms"]]

,host_id,name,price,room_type,beds,bedrooms
30,30193,Midtown Pied-a-terre,175.0,Entire home/apt,1.0,NaN
91,120223,East Village Sanctuary,100.0,Entire home/apt,2.0,NaN
478,1981742,Stylish Designer Studio with Piano,130.0,Entire home/apt,1.0,NaN
14894,33452526,Cozy apartment on the Upper East aside,250.0,Entire home/apt,1.0,NaN
15487,22345670,My Beach house,125.0,Private room,1.0,NaN
15955,86368678,Amazing modern place near JFK,60.0,Private room,1.0,NaN
16292,51548122,138 Bowery-Modern Queen Studio,150.0,Entire home/apt,1.0,NaN
16293,51548122,Cozy Studio Apartment,110.0,Private room,1.0,NaN
17580,55948559,XL Comfort Room.,43.0,Private room,1.0,NaN
19064,384532776,Beautiful & modern studio doorman elevator w/d...,110.0,Entire home/apt,2.0,NaN


In [8]:
clean_df.loc[clean_df["bedrooms"].isna(), "room_type"].value_counts(dropna=False)

room_type
Entire home/apt    28
Shared room        16
Private room        5
Name: count, dtype: int64

In [9]:
# 'bedrooms' 전처리 완
clean_df = clean_df.dropna(subset=["bedrooms"])
display(clean_df['bedrooms'].isna().sum())
display(clean_df['bedrooms'].value_counts(dropna=False))

np.int64(0)

bedrooms
1.0     14132
2.0      3863
0.0      2138
3.0      1444
4.0       449
5.0       156
6.0        53
7.0        13
9.0         5
8.0         4
14.0        1
15.0        1
Name: count, dtype: int64

In [10]:
# 'bed' 전처리
display(clean_df['beds'].isna().sum())
display(clean_df['beds'].value_counts(dropna=False))

np.int64(78)

beds
1.0     12962
2.0      5114
3.0      1896
4.0       911
0.0       634
5.0       328
6.0       180
NaN        78
7.0        66
8.0        54
9.0        17
10.0        7
12.0        4
13.0        3
11.0        2
42.0        1
14.0        1
21.0        1
Name: count, dtype: int64

In [11]:
clean_df.loc[clean_df["beds"].isna(), ["host_id", "name", "price", "room_type", "bedrooms", "beds"]]

,host_id,name,price,room_type,bedrooms,beds
15602,350773,NYU private quiet room in 1BR Apartment,161.0,Private room,1.0,NaN
15606,61391963,Polished Executive Apt Lexington Ave,142.0,Entire home/apt,0.0,NaN
15771,27107891,Live like a Brooklynite in heart of Williamsburg,143.0,Private room,1.0,NaN
15782,26924376,Luxury (Sky blue) 3 BED Apartment,150.0,Entire home/apt,3.0,NaN
15940,314493836,PRIVATE SAFE ROOMN QUIET FAMILY RESIDENTAL AREA,475.0,Private room,1.0,NaN
15966,204704622,Good Room New apartment in Williamsburg,48.0,Private room,1.0,NaN
16053,4373926,1 Bedroom in Harlem for the summer,50.0,Entire home/apt,1.0,NaN
16143,342548185,Near to the city,100.0,Private room,1.0,NaN
16417,236572440,Exquisite Single Bedroom in Shared Apartment,83.0,Private room,1.0,NaN
16767,30532557,Private Bedroom/ Retro large Apt 15 mins to City!,43.0,Private room,1.0,NaN


In [12]:
clean_df.loc[(clean_df["bedrooms"]>=1) & (clean_df["beds"] == 0),["host_id", "name", "price", "room_type", "bedrooms", "beds"]]

,host_id,name,price,room_type,bedrooms,beds
288,116599,The Brooklyn Waverly,800.0,Entire home/apt,1.0,0.0
406,2027013,"2RW - NEW RENO - EV STUDIO, WASHER/DRYER OFF ...",130.0,Entire home/apt,1.0,0.0
462,2647578,Large Bedroom with Modern Private Bathroom,120.0,Private room,1.0,0.0
489,61531,Red Hook Modern,126.0,Entire home/apt,1.0,0.0
623,1410714,Brooklyn Livin',92.0,Entire home/apt,1.0,0.0
...,...,...,...,...,...,...
37391,483056418,Furnished Private Room & Bath H,58.0,Private room,1.0,0.0
37410,107434423,"Blueground | Clinton Hill, pool, gym & AC, nr ...",298.0,Entire home/apt,1.0,0.0
37411,107434423,"Blueground | Clinton Hill, pool, gym & AC, nr ...",298.0,Entire home/apt,1.0,0.0
37430,483056418,Private Room w/ Ensuite Bath H,58.0,Private room,1.0,0.0


In [13]:
clean_df.loc[clean_df["beds"].isna(), "room_type"].value_counts(dropna=False)

room_type
Private room       57
Entire home/apt    21
Name: count, dtype: int64

In [14]:
clean_df.groupby("room_type")["beds"].median()

room_type
Entire home/apt    2.0
Hotel room         2.0
Private room       1.0
Shared room        1.0
Name: beds, dtype: float64

In [15]:
# 'beds' 전처리 완
clean_df["beds"] = clean_df["beds"].fillna(clean_df.groupby("room_type")["beds"].transform("median"))
clean_df["beds"].value_counts()

beds
1.0     13019
2.0      5135
3.0      1896
4.0       911
0.0       634
5.0       328
6.0       180
7.0        66
8.0        54
9.0        17
10.0        7
12.0        4
13.0        3
11.0        2
42.0        1
14.0        1
21.0        1
Name: count, dtype: int64

In [16]:
clean_df['beds'].isna().sum()

np.int64(0)

In [17]:
# 'review_scores_location' 전처리/ 나머지 평점이 모두 5인 행의 review_scores_location 컬럼이 null이므로 5로 대체
mask = (clean_df['review_scores_location'].isna()) & (clean_df['review_scores_accuracy'].notna())
clean_df.loc[mask, 'review_scores_location'] = 5.0

In [18]:
clean_df.loc[(clean_df['review_scores_location'].isna()) & (clean_df['review_scores_accuracy'].notna()),'review_scores_location']

Series([], Name: review_scores_location, dtype: float64)

In [19]:
# 'review_scores_location' 전처리 완 
print(clean_df.loc[(clean_df['review_scores_location'].isna())&(clean_df['review_scores_accuracy'].isna())].shape)
print(clean_df.loc[clean_df['review_scores_location'].isna()].shape)
print(clean_df.loc[clean_df['review_scores_accuracy'].isna()].shape)

(6772, 37)
(6772, 37)
(6772, 37)


In [20]:
# 'description' 전처리 : 리뷰 형식의 컬럼이므로 데이터가 없는 행의 null값만 결측 제거를 위해 unknown으로 대체
clean_df['description'].isna().sum()

np.int64(400)

In [21]:
clean_df['description'] = clean_df['description'].fillna('unknown')

In [22]:
# 'description' 전처리 완
print(clean_df['description'].isna().sum())
print(clean_df[clean_df['description'] == 'unknown'])

0
                        id                                               name  \
5                    39572                  1 br in a 2 br apt (Midtown West)   
17                   96471                  The Brooklyn Waverly, One Bedroom   
25                   42729  @HouseOnHenrySt - Private 2nd bedroom w/shared...   
40                   45910                 Beautiful Queens Brownstone! - 5BR   
42                   45936                 Couldn't Be Closer To Columbia Uni   
...                    ...                                                ...   
37291  1356862958825673292        A cozy private room in Brooklyn by the park   
37300  1362407813755960846     Ultra Luxury Furnished Flat for work & leisure   
37318  1357574158630690509                     Sunny and quiet 3rd Floor 1-BR   
37354  1358822659920606401                      New/Private 1 BR | Brownstone   
37380  1365330309802344714                    Charming & Cozy Home in Astoria   

      description    host

In [23]:
# 'host_since' 전처리 / host_name이 null 일 시 호스트 관련 행 전부 null, 그러나 다른 모든 행은 값이 있으므로
#  우선 unknown으로 처리 후 차후 상관관계, 통계적 유의성 확인 후 추가 전처리 진행
clean_df['host_since'].isna().sum()

np.int64(20)

In [24]:
clean_df['host_since'] = clean_df['host_since'].fillna('unknown')
clean_df['host_since'].isna().sum()

np.int64(0)

In [25]:
# 'host_since' 전처리 완
clean_df[clean_df['host_since'] == 'unknown']

,id,name,description,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,amenities,price,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
355,302758,Sunny 1BR East Harlem Apartment,Two-room apartment in East Harlem. Close to su...,8605,unknown,NaN,NaN,NaN,f,East Harlem,Manhattan,40.796750,-73.936340,Entire rental unit,Entire home/apt,2,1.0,2.0,"[""Air conditioning"", ""Wifi"", ""Smoke alarm"", ""K...",150.0,88,12,0,0,0.0,4.50,4.58,3.58,4.83,4.75,3.83,4.17,1,1,0,0,0.08
3528,7037918,"Near LGA, Citifield, US Open Tennis","Comfortable, sunny, cozy bedroom in private ho...",36242220,unknown,NaN,NaN,NaN,f,East Elmhurst,Queens,40.765500,-73.865770,Private room in home,Private room,2,1.0,1.0,"[""Fire extinguisher"", ""Luggage dropoff allowed...",70.0,365,229,0,0,0.0,4.77,4.87,4.76,4.91,4.93,4.87,4.76,1,0,1,0,1.97
8944,20193830,Family brownstone with backyard,"Located in Clinton Hill, a historic Brooklyn n...",1294005,unknown,NaN,NaN,NaN,t,Clinton Hill,Brooklyn,40.687851,-73.964070,Entire rental unit,Entire home/apt,5,3.0,4.0,"[""Portable fans"", ""Baking sheet"", ""Garden view...",614.0,178,113,34,255,156570.0,4.98,4.98,4.90,4.99,4.98,4.96,4.88,1,1,0,0,1.24
9561,21802923,Private Bedroom with Private Entrance in Brook...,You will have a large bedroom with a private e...,72466752,unknown,NaN,NaN,NaN,f,Bushwick,Brooklyn,40.701310,-73.917830,Private room in rental unit,Private room,1,1.0,1.0,"[""Oven"", ""Laundromat nearby"", ""Blender"", ""Dedi...",75.0,270,54,0,0,0.0,4.69,4.72,4.52,4.78,4.80,4.85,4.65,1,0,1,0,0.61
16572,45189535,Private Room for Sleeping in Great Neighborhood,"Safe, quiet, private, NYC room available just ...",364960386,unknown,NaN,NaN,NaN,f,Upper West Side,Manhattan,40.788340,-73.980060,Private room in rental unit,Private room,1,2.0,1.0,"[""Portable fans"", ""Laundromat nearby"", ""Luggag...",42.0,358,25,0,0,0.0,4.96,4.96,5.00,5.00,4.92,4.96,4.92,1,0,1,0,0.49
17341,47009616,1 bedroom apartment near Staten Island ferry,This is a one bedroom apartment with a living ...,260396179,unknown,NaN,NaN,NaN,f,Tompkinsville,Staten Island,40.634420,-74.087080,Entire rental unit,Entire home/apt,2,1.0,1.0,"[""Fire extinguisher"", ""Fast wifi \u2013 308 Mb...",77.0,80,24,1,60,4620.0,4.71,4.79,4.88,4.88,4.71,4.58,4.71,4,4,0,0,0.51
17398,46705156,2 Bedroom apartment in ENY Business District,Cozy 2BR APARTMENT with huge master BR and wal...,377484768,unknown,NaN,NaN,NaN,f,East New York,Brooklyn,40.668420,-73.899190,Entire home,Entire home/apt,5,2.0,3.0,"[""Oven"", ""Laundromat nearby"", ""Fire extinguish...",119.0,89,111,0,0,0.0,4.75,4.81,4.86,4.91,4.86,4.18,4.69,1,1,0,0,2.16
18705,49980221,2 bedroom apartment near Staten Island ferry,This is a two bedroom apartment near the State...,260396179,unknown,NaN,NaN,NaN,f,Tompkinsville,Staten Island,40.634880,-74.087110,Entire rental unit,Entire home/apt,5,2.0,2.0,"[""Oven"", ""Laundromat nearby"", ""First aid kit"",...",99.0,81,28,1,60,5940.0,4.82,4.86,4.71,4.93,4.96,4.57,4.71,4,4,0,0,0.76
20106,53050810,Bedroom w/private bathroom in Unit2 in Greenpoint,"Greenpoint is a safe, convenient option for ex...",17888612,unknown,NaN,NaN,NaN,t,Greenpoint,Brooklyn,40.728110,-73.947690,Private room in townhouse,Private room,2,1.0,1.0,"[""Oven"", ""Laundromat nearby"", ""Fire extinguish...",123.0,75,172,47,255,31365.0,4.94,4.98,4.82,4.94,4.99,4.95,4.93,1,0,1,0,4.30
24325,736139385311072356,Relaxing 1 Bedroom in Brooklyn,"Unwind in this cute, moder

In [26]:
# 'host_response_time' 전처리 / 응답률이 빠를 수록 순위를 두기 위해 순위형 데이터로 매핑 이후 높은 비율의 무응답률이 null 처리 되어
# 있어 자료형을 맞추기 위해 우선 -1로 변환 차후 필요 시 -1을 무응답으로 대입해서 활용하거나 자료형 변환 예정
clean_df['host_response_time'].value_counts()

host_response_time
within an hour        11615
within a few hours     3429
within a day           1954
a few days or more      872
Name: count, dtype: int64

In [27]:
clean_df['host_response_time'].isna().sum()

np.int64(4389)

In [28]:
mapping = {'within an hour': 4, 'within a few hours': 3, 'within a day': 2, 'a few days or more': 1}
clean_df['host_response_time'] = clean_df['host_response_time'].map(mapping)
clean_df['host_response_time'].value_counts()

host_response_time
4.0    11615
3.0     3429
2.0     1954
1.0      872
Name: count, dtype: int64

In [29]:
# 'host_response_time' 전처리 완
clean_df['host_response_time'] = clean_df['host_response_time'].fillna(-1)
clean_df['host_response_time'].value_counts()

host_response_time
 4.0    11615
-1.0     4389
 3.0     3429
 2.0     1954
 1.0      872
Name: count, dtype: int64

In [30]:
# 'host_response_rate' 전처리 / host_response_time 즉 응답시간이 없는 행은 host_response_rate 응답률도 같이 없기 때문에
# 같은 형식으로 null값 -1로 전처리 추후 같은 방법 활용 예정
clean_df['host_response_rate'].isna().sum()

np.int64(4389)

In [31]:
clean_df['host_response_rate'] = (clean_df['host_response_rate'].str.replace('%', '', regex=False).astype(float))
clean_df['host_response_rate'].value_counts()

host_response_rate
100.0    12861
99.0       727
98.0       549
0.0        436
90.0       349
68.0       305
80.0       215
97.0       196
92.0       182
42.0       174
50.0       173
94.0       150
91.0       129
96.0       126
95.0       103
75.0        94
67.0        90
79.0        88
93.0        80
48.0        78
89.0        68
60.0        65
83.0        63
86.0        57
88.0        53
70.0        50
82.0        35
33.0        30
87.0        28
25.0        26
78.0        24
85.0        24
40.0        22
20.0        22
81.0        21
41.0        21
30.0        20
57.0        18
71.0        15
56.0        15
10.0         8
74.0         7
38.0         7
53.0         6
65.0         6
17.0         5
34.0         5
76.0         5
36.0         5
63.0         4
46.0         4
61.0         4
77.0         3
14.0         2
44.0         2
58.0         2
29.0         2
84.0         2
72.0         2
73.0         2
16.0         1
43.0         1
55.0         1
11.0         1
62.0         1
Name: 

In [32]:
clean_df['host_response_rate'] = clean_df['host_response_rate'].fillna(-1)
clean_df['host_response_rate'].isna().sum()

np.int64(0)

In [33]:
# 'host_response_rate' 전처리 완
clean_df['host_response_rate'].value_counts()

host_response_rate
 100.0    12861
-1.0       4389
 99.0       727
 98.0       549
 0.0        436
 90.0       349
 68.0       305
 80.0       215
 97.0       196
 92.0       182
 42.0       174
 50.0       173
 94.0       150
 91.0       129
 96.0       126
 95.0       103
 75.0        94
 67.0        90
 79.0        88
 93.0        80
 48.0        78
 89.0        68
 60.0        65
 83.0        63
 86.0        57
 88.0        53
 70.0        50
 82.0        35
 33.0        30
 87.0        28
 25.0        26
 78.0        24
 85.0        24
 40.0        22
 20.0        22
 81.0        21
 41.0        21
 30.0        20
 57.0        18
 71.0        15
 56.0        15
 10.0         8
 74.0         7
 38.0         7
 53.0         6
 65.0         6
 17.0         5
 34.0         5
 76.0         5
 36.0         5
 63.0         4
 46.0         4
 61.0         4
 77.0         3
 14.0         2
 44.0         2
 58.0         2
 29.0         2
 84.0         2
 72.0         2
 73.0         2
 16.0

In [34]:
# 'host_acceptance_rate' 전처리 • host_acceptance_rate : 예약 요청 수락률은 null도 null의 의미로 존재할 필요성을 예상
clean_df['host_acceptance_rate'].isna().sum()

np.int64(3463)

In [35]:
# # 'host_acceptance_rate' 전처리 완
clean_df['host_acceptance_rate'] = (clean_df['host_acceptance_rate'].str.replace('%', '', regex=False).astype(float))
clean_df['host_acceptance_rate'].describe()

count    18796.000000
mean        78.151522
std         27.948701
min          0.000000
25%         67.000000
50%         91.000000
75%        100.000000
max        100.000000
Name: host_acceptance_rate, dtype: float64

In [36]:
# 'host_is_superhost' 전처리 / 374개의 결측 값은 이후 머신러닝에서 활용할 때는 전체비율 대비 결측데이터가 적어 
# drop 처리를 할 예정 이지만 통계 검정 등의 활용 시 다음과 같이 전처리를 해서 데이터 손실을 줄일 예정
print(clean_df['host_is_superhost'].value_counts())
print(clean_df['host_is_superhost'].isna().sum())

host_is_superhost
f    15756
t     6129
Name: count, dtype: int64
374


In [37]:
clean_df['host_is_superhost'] = clean_df['host_is_superhost'].map({'t':True, 'f':False})
clean_df.loc[
    (clean_df["host_is_superhost"] == 1) & 
    (clean_df["host_acceptance_rate"].isna())
]

,id,name,description,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,amenities,price,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
8944,20193830,Family brownstone with backyard,"Located in Clinton Hill, a historic Brooklyn n...",1294005,unknown,-1.0,-1.0,NaN,True,Clinton Hill,Brooklyn,40.687851,-73.964070,Entire rental unit,Entire home/apt,5,3.0,4.0,"[""Portable fans"", ""Baking sheet"", ""Garden view...",614.0,178,113,34,255,156570.0,4.98,4.98,4.90,4.99,4.98,4.96,4.88,1,1,0,0,1.24
20106,53050810,Bedroom w/private bathroom in Unit2 in Greenpoint,"Greenpoint is a safe, convenient option for ex...",17888612,unknown,-1.0,-1.0,NaN,True,Greenpoint,Brooklyn,40.728110,-73.947690,Private room in townhouse,Private room,2,1.0,1.0,"[""Oven"", ""Laundromat nearby"", ""Fire extinguish...",123.0,75,172,47,255,31365.0,4.94,4.98,4.82,4.94,4.99,4.95,4.93,1,0,1,0,4.30
20171,53155469,Cozy Private Bedroom Queens Village - Shared Apt,Make yourself at home! Relax & enjoy your stay...,52155780,2015-12-24,-1.0,-1.0,NaN,True,Queens Village,Queens,40.707850,-73.746790,Private room in rental unit,Private room,1,1.0,1.0,"[""Cooking basics"", ""Self check-in"", ""Essential...",60.0,90,20,1,60,3600.0,5.00,5.00,5.00,5.00,5.00,4.75,5.00,1,0,1,0,0.50
24325,736139385311072356,Relaxing 1 Bedroom in Brooklyn,"Unwind in this cute, modern, 1 bedroom apartme...",35174210,unknown,-1.0,-1.0,NaN,True,Bay Ridge,Brooklyn,40.629113,-74.028760,Entire rental unit,Entire home/apt,2,1.0,1.0,"[""Baking sheet"", ""Fire extinguisher"", ""First a...",94.0,256,6,5,255,23970.0,5.00,4.83,5.00,5.00,5.00,5.00,4.83,2,2,0,0,0.32
26989,868140563222071942,Penthouse walking to Best of NYC,Enjoy this sunny south facing centrally-locate...,50402335,unknown,-1.0,-1.0,NaN,True,Kips Bay,Manhattan,40.741667,-73.980570,Entire rental unit,Entire home/apt,4,1.0,2.0,"[""Portable fans"", ""Laundromat nearby"", ""Fire e...",207.0,335,1,0,0,0.0,5.00,5.00,5.00,5.00,5.00,5.00,5.00,1,1,0,0,0.07
29443,982833752250568153,Comfortable Brownstone Retreat,Spacious tasteful designer brownstone suite we...,220765580,unknown,-1.0,-1.0,NaN,True,Bedford-Stuyvesant,Brooklyn,40.680260,-73.951640,Private room in townhouse,Private room,2,2.0,2.0,"[""Laundromat nearby"", ""Fire extinguisher"", ""Lu...",160.0,97,23,23,255,40800.0,5.00,5.00,4.83,5.00,5.00,4.83,4.91,1,0,1,0,3.79
33845,1181394579984036414,A cozy apartment near Yankee Stadium,"We are a 20min drive from LGA Airport, and 15m...",252043685,unknown,-1.0,-1.0,NaN,True,Melrose,Bronx,40.819851,-73.915504,Private room in townhouse,Private room,4,2.0,2.0,"[""Oven"", ""Rice maker"", ""Fire extinguisher"", ""L...",135.0,121,11,11,141,19035.0,4.91,4.91,4.82,5.00,5.00,4.64,4.91,1,0,1,0,1.90


In [38]:
clean_df['host_is_superhost'] = clean_df['host_is_superhost'].fillna('unknown')
clean_df[clean_df['host_is_superhost'] == 'unknown']

,id,name,description,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,amenities,price,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
401,333323,Cozy bedroom near Times Square,1 bedroom in a 3 bedroom apartment centrally l...,1698391,2012-02-04,4.0,100.0,89.0,unknown,Hell's Kitchen,Manhattan,40.762690,-73.993610,Private room in rental unit,Private room,1,1.0,1.0,"[""Oven"", ""Cooking basics"", ""Microwave"", ""Hair ...",82.0,238,185,6,255,20910.0,4.75,4.81,4.58,4.86,4.84,4.94,4.71,3,1,2,0,1.48
668,783202,Charming First Floor Village Apt off Bleecker,DO NOT REQUEST A BOOKING TIL WE CHAT! SCROLL ...,4129805,2012-11-12,4.0,98.0,69.0,unknown,West Village,Manhattan,40.732770,-74.002670,Entire rental unit,Entire home/apt,3,1.0,2.0,"[""Oven"", ""Babysitter recommendations"", ""Dedica...",182.0,56,402,51,255,46410.0,4.73,4.73,4.68,4.85,4.80,4.98,4.72,5,5,0,0,2.69
715,786685,Historic Brownstone dwelling gorgeous garden,"Beautifully spacious rooms with huge, secure w...",4147380,2012-11-14,4.0,100.0,100.0,unknown,Harlem,Manhattan,40.822930,-73.949540,Entire townhouse,Entire home/apt,6,1.0,2.0,"[""Portable fans"", ""Oven"", ""Garden view"", ""Baki...",270.0,255,217,1,60,16200.0,4.89,4.89,4.93,4.96,4.96,4.76,4.80,1,1,0,0,1.57
873,1042806,My Other Little Guestroom,Warm and Cozy newly renovated bedroom in a fam...,2680820,2012-06-19,4.0,100.0,97.0,unknown,Flushing,Queens,40.755331,-73.817833,Private room in home,Private room,2,1.0,1.0,"[""Fire extinguisher"", ""First aid kit"", ""Keypad...",81.0,109,416,55,255,20655.0,4.80,4.84,4.80,4.92,4.89,4.61,4.81,1,0,1,0,2.87
912,1411811,"Charming Parlor Apt by Bleecker, Old World Vil...",PLS BE CONSIDERATE! Your request blocks my ca...,4129805,2012-11-12,4.0,98.0,69.0,unknown,West Village,Manhattan,40.731000,-74.003460,Entire rental unit,Entire home/apt,2,1.0,1.0,"[""Oven"", ""Babysitter recommendations"", ""Dedica...",302.0,48,318,35,255,77010.0,4.81,4.83,4.77,4.87,4.85,4.97,4.78,5,5,0,0,2.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37160,1359702695327440866,Large Modern Bedroom 1-B,You are looking at a gorgeous large size bedro...,371046050,2020-10-07,4.0,100.0,50.0,unknown,Edenwald,Bronx,40.892832,-73.840800,Private room in home,Private room,2,1.0,1.0,"[""Oven"", ""Rice maker"", ""Laundromat nearby"", ""D...",40.0,337,0,0,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
37161,1359754329005822690,spacious 3 bedrooms in the city,This stylish place to stay is perfect for grou...,257606572,2019-04-23,4.0,100.0,100.0,unknown,East Harlem,Manhattan,40.795017,-73.947907,Entire rental unit,Entire home/apt,8,3.0,3.0,"[""First aid kit"", ""Dedicated workspace"", ""Extr...",704.0,365,0,0,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,0,0,NaN
37278,1364579260115677851,89-2R 1BR/1Bath on the UES,Embark on an unforgettable journey into the he...,29468219,2015-03-16,2.0,67.0,42.0,unknown,Upper East Side,Manhattan,40.780048,-73.950582,Entire rental unit,Entire home/apt,2,1.0,1.0,"[""Oven"", ""Rice maker"", ""Blender"", ""Dedicated w...",124.0,351,0,0,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14,14,0,0,NaN
37408,1365681782045610391,Private NYC Bedroom,You'll have a great time at this comfortable p...,314462294,2019-12-04,2.0,85.0,14.0,unknown,Woodside,Queens,40.746699,-73.907028,Private room in rental unit,Private room,2,1.0,1.0,"[""Lock on bedroom door"", ""Fire 

In [39]:
# host_is_superhost 기준으로 그룹을 나누고 각 그룹의 host_acceptance_rate 중앙값으로 결측값을 대체
# 슈퍼호스트와 일반호스트 그룹을 나누어 각 그룹의 중앙값으로 host_acceptance_rate 결측을 채움.
clean_df['host_acceptance_rate'] = (
    clean_df.groupby('host_is_superhost')['host_acceptance_rate']
       .transform(lambda x: x.fillna(x.median()))
)

In [40]:
clean_df['host_acceptance_rate'].isna().sum()

np.int64(0)

In [41]:
display(clean_df['host_acceptance_rate'].value_counts())

host_acceptance_rate
100.0    4891
87.0     3559
97.0     1407
99.0     1006
0.0       997
83.0      473
77.0      473
88.0      469
50.0      451
96.0      422
98.0      384
67.0      377
91.0      374
49.0      331
82.0      290
84.0      270
60.0      252
93.0      250
75.0      246
94.0      244
86.0      242
95.0      239
47.0      227
80.0      219
92.0      218
89.0      196
90.0      177
72.0      169
33.0      156
71.0      143
70.0      137
24.0      135
73.0      127
51.0      126
79.0      112
81.0      110
78.0      106
55.0       99
58.0       98
57.0       96
85.0       87
56.0       84
37.0       79
25.0       78
74.0       76
40.0       75
48.0       70
69.0       67
59.0       67
52.0       64
68.0       62
65.0       59
46.0       58
41.0       56
61.0       56
76.0       53
63.0       52
43.0       51
62.0       49
39.0       49
20.0       48
64.0       48
53.0       46
29.0       43
30.0       39
17.0       35
44.0       31
8.0        30
23.0       27
27.0       25

In [42]:
# 'host_is_superhost' 1차 전처리 완
clean_df['host_is_superhost'].head()

0    False
1    False
2    False
3     True
4     True
Name: host_is_superhost, dtype: object

In [43]:
# 가격 2차 전처리 로그 변환 후 이상치 확인과 변환값 재 변환
clean_df['price'] = clean_df['price'].replace(r'[\$,]', '', regex=True).astype(float).astype(int)

clean_df['log_price'] = np.log1p(clean_df['price'])

q1_log = np.percentile(clean_df['log_price'], 25)
q3_log = np.percentile(clean_df['log_price'], 75)
iqr_log = q3_log - q1_log

lower_log = q1_log - 1.5 * iqr_log
upper_log = q3_log + 1.5 * iqr_log

log_outliers = clean_df.loc[(clean_df['log_price'] > upper_log) | (clean_df['log_price'] < lower_log)]

print(f"로그 변환 후 이상치 개수: {len(log_outliers)}")

로그 변환 후 이상치 개수: 252


In [44]:
# lower 바운드의 이상치 제거 upper 바운드의 이상치는 활용 가능성이 있으므로 우선은 분리 뒤 추가 이상치 제거 박스에 붙임처리
clean_df = clean_df.loc[(clean_df['log_price'].between(lower_log, upper_log))+(clean_df['log_price']>upper_log)]
clean_df['price'].describe()

count    22248.000000
mean       214.138574
std        428.107042
min         19.000000
25%         85.000000
50%        140.000000
75%        240.000000
max      20000.000000
Name: price, dtype: float64

In [45]:
# 활용을 위한 1차 전처리 데이터 = clean_df
clean_df.shape

(22248, 38)

In [46]:
clean_df.to_csv("data/first_clean_data")